# Bybit Spot Historical Data Import

This notebook imports historical OHLCV data from Bybit Spot markets into the PostgreSQL database.

**Data Source:** Bybit V5 API (https://api.bybit.com/v5)

**Features:**
- Fetches historical klines/candlestick data from Bybit API
- Automatic pagination for large date ranges
- Automatic duplicate detection
- Bulk insert operations
- Includes 2025 data (unlike public file repository)

**Data Format:**
- API returns: [timestamp_ms, open, high, low, close, volume, turnover]
- Timestamps are in milliseconds
- Max 1000 records per request
- Data returned in reverse chronological order (converted to chronological)

**Available Symbols:**
All Bybit spot trading pairs (BTCUSDT, ETHUSDT, etc.)

**API Limits:**
- Public endpoints: 50 requests/second per IP
- Max 1000 klines per request
- Supports historical data from 2020 onwards
- 2025 data available via API (not in public files)

**Notes:**
- API method is more flexible than public files
- Can fetch any date range including current data
- Automatic retry logic with exponential backoff
- Rate limiting: 20 req/s (conservative, well below 50 req/s limit)

In [1]:
import pandas as pd
import time

from clients.db_client import DBClient
from clients.bybit_client import BybitClient

# Configuration
INTERVAL = "60"  # Options: 1, 3, 5, 15, 30, 60, 120, 240, 360, 720, D, W, M
START_DATE = "2023-01-01"
END_DATE = "2025-12-31"

# Initialize clients
db = DBClient()
bybit = BybitClient(request_delay=0.05, max_retries=3)

In [ ]:
# Test download
test_df = bybit.download_klines("BTCUSDT", INTERVAL, "2023-01-01", "2025-12-31")
print(f"\nFirst few rows:")
print(test_df.head())
print(f"\nLast few rows:")
print(test_df.tail())
print(f"\nDate range: {test_df['timestamp'].min()} to {test_df['timestamp'].max()}")
print(f"Total rows: {len(test_df)}")

In [2]:
# Import Bybit data
total_inserted = 0
total_skipped = 0

# Get all Bybit instruments from database
instruments_dict = db.get_instruments_by_exchange(exchange="bybit")
print(f"Found {len(instruments_dict)} Bybit instruments in database\n")

if not instruments_dict:
    print("No instruments found..")

for symbol in instruments_dict.keys():
    print(f"\n{'='*60}")
    print(f"Processing {symbol} (instrument_id: {instruments_dict[symbol]})")
    print(f"{'='*60}")
    
    instrument_id = instruments_dict[symbol]
    
    # Download all data for date range
    df = bybit.download_klines(symbol, INTERVAL, START_DATE, END_DATE)
    
    if df.empty:
        print(f"No data available from {START_DATE} - {END_DATE}!")
        continue
    
    # Get existing timestamp range from database
    db_min, db_max = db.get_timestamp_range(instrument_id)
    
    # Check for duplicates
    if db_min and db_max:
        # Filter out overlapping data
        df = df[(df['timestamp'] < db_min) | (df['timestamp'] > db_max)]
        
        if df.empty:
            print(f"All data already exists, skipping")
            total_skipped += 1
            continue
    
    # Insert data
    df['instrument_id'] = instrument_id
    rows = db.insert_market_data(df)
    total_inserted += rows
    print(f"Inserted {rows} rows")

print(f"\n{'='*60}")
print("IMPORT COMPLETE")
print(f"{'='*60}")
print(f"Total rows inserted: {total_inserted:,}")
print(f"Total symbols skipped (duplicates): {total_skipped}")

Found 23 Bybit instruments in database


Processing BTCUSDT (instrument_id: 41)
Fetched 10000 candles (page 10)...
Fetched 20000 candles (page 20)...
Downloaded 26304 candles (27 API calls)
Inserted 26304 rows

Processing ETHUSDT (instrument_id: 42)
Fetched 10000 candles (page 10)...
Fetched 20000 candles (page 20)...
Downloaded 26304 candles (27 API calls)
Inserted 26304 rows

Processing BNBUSDT (instrument_id: 43)
Fetched 10000 candles (page 10)...
Fetched 20000 candles (page 20)...
Downloaded 26304 candles (27 API calls)
Inserted 26304 rows

Processing SOLUSDT (instrument_id: 44)
Fetched 10000 candles (page 10)...
Fetched 20000 candles (page 20)...
Downloaded 26304 candles (27 API calls)
Inserted 26304 rows

Processing XRPUSDT (instrument_id: 45)
Fetched 10000 candles (page 10)...
Fetched 20000 candles (page 20)...
Downloaded 26304 candles (27 API calls)
Inserted 26304 rows

Processing ADAUSDT (instrument_id: 46)
Fetched 10000 candles (page 10)...
Fetched 20000 candles (page 20)...
